In [ ]:
"""
Prosjektoppgave PY1010 ved USN April 2026

av Geir Olav Kaase
geirolavkaase@gmail.com

Rev: 14.03.2026

Beskrivelse:

    Scriptet generere inputkode for programmet STAAD.PRO
    STAAD.PRO er et program brukt for kodesjekk / dimensjonering av bjelke-elementer i stålkonstruksjoner.

    Dette Python-scriptet tar input fra bruker på byggets ytre dimensjoner og verdi på snølast for så å generere 
    input-kommandoer for analyse i Staad.Pro. Kodesjekk utføres etter NS-EN1993, som er kravet for alle bygg i Norge. 
    Material er også satt til S355-stål, men dette er egentlig ingen praktisk begrensning.
   
    Foreløpig er profiltype hardkodet. 
    Naturlig fortsettelse vil være å lage GUI som gir bruker mulighet til å velge forskjellige profiltyper for de ulike bjelke-elementene.

    Scriptet er utviklet for å raskt kunne modellere og utføre kapasitetssjekk for "enkle" stål-bygg som garasjer og næringsbygg.
    Disse vil ha en utforming som beskrevet i dette scriptet.

    Med scriptet vil en jobb som ellers tar 1 time ta "2 minutter".
    Videre utvikling ved å implementere alle profiltyper og kontroll av utnyttelsesgrad og iterasjon på ny profil-type kunne
    optimalisere et bygg, og redusere en jobb fra 4-6 timers ned til få minutter. 

Om koden:

    Koden ble først gradvis utviklet som flere moduler, og er tenkt kjørt som moduler i praktisk bruk.
    Her er modulene satt sammen på "enkleste" vis for å samle alt til en kode-fil, ifb med innlevering av prosjektoppgave.

Koden består av følgende deler, basert på rekkefølgekrav i Staad-input fil:

Del #1 - GUI geometri: tar bruker input på byggets dimensjoner og takoppbygning
Del #2 - Geometri: bygger noder og bjelkeelementer
Del #3 - Material: definerer materialegenskaper, fastsatt til høyfast stål S355 for alle materialer. 
Del #4 - Profiler: definerer bjelkenes tverrsnittstype basert på GUI for bruker input (forløpig begrenset bibliotek som kan utvides)
Del #5 - Support: definerer opplagerbetingelser, fastsatt til fritt opplager i alle søyler. 
Del #6 - Laster: definerer snø på taket av bygget iht til NS-EN 1990, med bruker GUI for valg av kommune og parametre
Del #7 - Kodesjekk: utfører kodesjekk etter NS-EN 1993 (Eurocode 3)
Del #8 - STD-fil: skriver en komplett Staad.Pro std-file som kan brukes til å kjøre en kodesjekk av bygget: staad-command.std
Del #9 - Plotting: plotter byggets noder, mao knutepunkter mellom bjelker: fungerer som enkel kontroll av input

Kontroll:

Dersom man spesifiserer et bygg med B=4, L=8, H=3, antall takstoler = 12, møne = ja, mH=5 og en snølast på 250, vil man generere en 
staad-command.std fil lik den som er lastet opp i Github. Når denne så kjøres av Staad.Pro, suksessfullt, vil man få en output.fil like den anl-fil som er lastet 
opp i Github. I begynnelsen av denne filen gjentas koden i input-filen, og man kan dermed verifisere at den koden som dette scriptet genererer
faktisk vil resultere i en feilfri analyse i Staad.Pro.

__________________________________________________________________________________________________________________________
Del #1 - Geometri GUI

Genererer GUI til geometri input fra bruker om byggets dimensjoner
(Opprinnelig skrevet som selvstendig modul - men modifisert enklest mulig for å fungere i en fil ifb med denne innlevering)
_________________________________________________________________________________________________________________________
"""
# Importerer tkinter for å lage grafisk grensesnitt
import tkinter as tk
from tkinter import ttk, messagebox

# Setter opp funksjoner som kan konvertere tekst (streng) input til flyttall eller heltall. 
def parse_float(s: str):
    """Konverterer streng til float, aksepterer komma og punktum."""
    try:
        return float(str(s).replace(",", ".").strip())
    except (AttributeError, ValueError):
        return None

def parse_int(s: str):
    """Konverterer streng til heltall. Returnerer None ved ugyldig input."""
    try:
        return int(str(s).strip())
    except (TypeError, ValueError):
        return None

#_____________________________________________________________________________________________________________________
# Setter opp GUI-funksjonen
#_____________________________________________________________________________________________________________________

def open_gui_and_get_data():
    """
    Åpner GUI (blokkerer til lukket), validerer, og returnerer et dict med ALLE felt:
      - B, L, H, mH (floats)
      - n_truss (int)
      - mone (str: ja eller /'nei)
    """
    # Oppretter GUI vinduet med tittel, og størrelse som kan utvides
    root = tk.Tk()
    root.title("Byggdata")
    main = ttk.Frame(root, padding=12)
    main.grid(row=0, column=0, sticky="nsew")
    root.columnconfigure(0, weight=1)
    root.rowconfigure(0, weight=1)
   
    # Legger til forklarende tekst
    ttk.Label(
        main,
        text="Vennligst oppgi mål for å beskrive byggets ytre dimensjoner og oppbygging"
    ).grid(row=0, column=0, columnspan=2, sticky="w", pady=(0, 8))

    # Oppretter Tk-variabler med bruker-input som tekst fra bruker
    B_var = tk.StringVar()
    L_var = tk.StringVar()
    H_var = tk.StringVar()
    mone_var = tk.StringVar(value="Nei") # Setter startverdi til "nei"
    mH_var = tk.StringVar()
    truss_var = tk.StringVar()

    # Oppretter input felt for variablene med tekst beskrivelse i GUI:
    ttk.Label(main, text="Bredde (B) i meter:").grid(row=1, column=0, sticky="e", padx=(0, 8))
    ttk.Entry(main, textvariable=B_var, width=20).grid(row=1, column=1, sticky="w")

    ttk.Label(main, text="Lengde (L) i meter:").grid(row=2, column=0, sticky="e", padx=(0, 8))
    ttk.Entry(main, textvariable=L_var, width=20).grid(row=2, column=1, sticky="w")

    ttk.Label(main, text="Gesimshøyde (H) i meter:").grid(row=3, column=0, sticky="e", padx=(0, 8))
    ttk.Entry(main, textvariable=H_var, width=20).grid(row=3, column=1, sticky="w")

    # Oppretter input felt for antall takstoler/rammer:
    ttk.Label(
        main,
        text="Antall takstoler/rammer i tak (eksl. mønevegger):"
    ).grid(row=4, column=0, sticky="e", padx=(0, 8))
    ttk.Entry(main, textvariable=truss_var, width=20).grid(row=4, column=1, sticky="w")

    # Oppretter input felt for mønet:
    ttk.Label(main, text="Har bygget møne? (Ja/Nei):").grid(row=5, column=0, sticky="e", padx=(0, 8))
    ttk.Entry(main, textvariable=mone_var, width=20).grid(row=5, column=1, sticky="w")

    # Oppretter felt for inout av mønehøyde, som vises kun hvis mønet settet til ja
    mH_label = ttk.Label(main, text="Mønehøyde (mH) i meter:")
    mH_entry = ttk.Entry(main, textvariable=mH_var, width=20)

    # Viser eller skjuler felt for mønehøyde om ja
    def toggle_monehoyde(*_):
        if mone_var.get().strip().lower() == "ja": # henter input, fjerne mellomrom og gjør om til små bokstaver
            mH_label.grid(row=6, column=0, sticky="e", padx=(0, 8)) # legger felt for input til i GUI
            mH_entry.grid(row=6, column=1, sticky="w")
        else:
            mH_label.grid_remove() # skjuler input feltet om bruker svarer nei
            mH_entry.grid_remove()
  
    # Bruker funksjonen for å vise eller skjule felt for mønehøyde
    mone_var.trace_add("write", toggle_monehoyde)
    toggle_monehoyde()  # initierer funksjonen med mønet satt til nei (mone_var) som startverdi

    # Oppretter dictionary for å fylle inn input-verdier, både som tekst og konvertert tallformat
    result = {
        # Variable klare for bruk
        "B": None, "L": None, "H": None, "mH": None,
        "n_truss": None, "mone": None,
    }

    # Oppretter funksjon som behandler input data sendt av bruker i GUIet
    def submit():
        # Leser input-verdier fra GUIet
        B_txt = B_var.get()
        L_txt = L_var.get()
        H_txt = H_var.get()
        n_truss_txt = truss_var.get()
        mone_txt = mone_var.get()
        mH_txt = mH_var.get()

        # Konverterer tekst til flyttall 
        B = parse_float(B_txt)
        L = parse_float(L_txt)
        H = parse_float(H_txt)

        if B is None or B <= 0:
            messagebox.showerror("Feil", "Oppgi gyldig, positiv bredde (B) i meter.")
            return
        if L is None or L <= 0:
            messagebox.showerror("Feil", "Oppgi gyldig, positiv lengde (L) i meter.")
            return
        if H is None or H <= 0:
            messagebox.showerror("Feil", "Oppgi gyldig, positiv gesimshøyde (H) i meter.")
            return
            
        # Konverterer tekst til heltall og gir feilmelding om < 0
        n_truss = parse_int(n_truss_txt)
        if n_truss is None or n_truss < 0:
            messagebox.showerror(
            "Feil",
            "Oppgi et gyldig heltall ≥ 0 for antall takstoler/rammer (eksl. mønevegger)."
            )
            return
        # Konverterer tekst til flyttall for mønehøyde og gir feilmeldinger ved feil input
        mone = str(mone_txt).strip().lower()
        mH_val = None
        if mone == "ja":
            mH_val = parse_float(mH_txt)
            if mH_val is None:
                messagebox.showerror("Feil", "Oppgi gyldig mønehøyde (mH) i meter.")
                return
            if mH_val <= H:
                messagebox.showerror("Feil", "mH må være større enn H.")
                return
        elif mone != "nei":
            messagebox.showerror("Feil", "Skriv 'Ja' eller 'Nei' for møne.")
            return

        # Oppsummerer input i eget vindu for kontroll av bruker
        lines = [
            f"B={B} m, L={L} m, H={H} m",
            f"Antall takstoler/rammer (eksl. mønevegger): {n_truss}",
            "Møne: " + ("Ja, mH=" + str(mH_val) + " m" if mone == "ja" else "Nei")
        ]
        # Viser i info boks
        messagebox.showinfo("OK", "\n".join(lines))

        # Oppdaterer og lagrer result-dict og lukker GUI vinduet tilslutt
        result.update({
            "B": B, "L": L, "H": H, "mH": mH_val,
            "n_truss": n_truss, "mone": mone,
        })
        root.destroy() #Lukker GUI

    # Oppretter knapp for bruker å "sende inn" angitte verdier, og kaller funksjonen submit() definert ovenfor
    ttk.Button(main, text="Send inn", command=submit).grid(row=8, column=0, columnspan=2, pady=(12, 0))

    # Sender inn data når bruker trykker "Enter", isteden for å trykke inn "Send inn"
    root.bind("<Return>", lambda _e: submit())

    # Starter tkinter hovedløkken, viser vindet og venter på bruker input
    root.mainloop()
    # Når vinduet er lukket returneres result-dict med gitte verdier fra bruker
    return result

#___________________________________________________________________________________________________________________________
#  Åpner GUI ved import av modulen og gjør variablene tilgjengelig på modul nivå
#___________________________________________________________________________________________________________________________

# Åpner GUI straks modulen importeres.
# Lagrer innlest data i dict: _data
_data = open_gui_and_get_data()

# Gjør ALLE input-variablene direkte tilgjengelige:
B         = _data["B"]
L         = _data["L"]
H         = _data["H"]
mH        = _data["mH"]
n_truss   = _data["n_truss"]
mone      = _data["mone"]

# Rydder opp, sletter intern variabel
del _data

"""
_____________________________________________________________________________________________________________________________
Del #2 - Geometri kommandoer for Staad

Genererer geometri data til Staad.Pro basert på input fra bruker om byggets dimensjoner
Samler Staad geometri-kode i filen: staad-geometri.txt
(Opprinnelig skrevet som selvstendig modul - men modifisert enklest mulig for å fungere i en fil ifb med denne innlevering)
_____________________________________________________________________________________________________________________________

Skriver header / standard start-kommanoder for STAAD til filen: staad-command.txt
"""
# Henter dagens dato i ønsket format (DD-MM-YY)
from datetime import datetime
dato = datetime.now().strftime("%d-%b-%y")

# Etablerer STAAD command file header
header = [
    "STAAD SPACE",
    "START JOB INFORMATION",
    f"ENGINEER DATE {dato}",
    "END JOB INFORMATION",
    "INPUT WIDTH 79",
    "UNIT METER KN"
]
with open("staad-geometri.txt", "w", encoding="utf-8") as f:
    for linje in header:
        f.write(linje + "\n")

#__________________________________________________________________________________________________________________________
# Definerer koordinater på noder og skriver STAAD kommandoer til txt-fil
#_________________________________________________________________________________________________________________________

with open("staad-geometri.txt", "a", encoding="utf-8") as f:
    print("JOINT COORDINATES", file=f)

# Noder i fundament (y=0 in STAAD)
    nodes0 = {
        1: [0, 0, 0],
        2: [0, 0, B],
        3: [L, 0, 0],
        4: [L, 0, B]
    }
    for i, verdier in nodes0.items():
        print(i, *verdier, ";", file=f)

# Setter opp funksjon for å bygge dictionary med noder i gesimshøyde /etasjeskille med tilhørende koordinater
def dict_noder(n_truss, start, Y, Z1, Z2,):
    """
    Lager dictionary med nodes 
    n_truss: noder i x-akse = antall takstoler mellom mønevegger
    Y: y-koordinat, Y blir Z i Staad 
    Z: z-koordinat, Z blir Y i Staad (vertikal akse)
    """
    noder = {}
    node_index = start  # Startverdi for dictionary indeksnummerering
    ts = L / (n_truss - 1) # nodeavstand (takbjelke/takstol avstand)

    for i in range(n_truss):
        TSx = ts*i  

        # Node ved z = Z1: dvs første langveggen
        noder[node_index] = [TSx, Y, Z1] #Y blir Z i Staad
        node_index += 1

        # Node ved z = Z2: dvs andre langveggen
        noder[node_index] = [TSx, Y, Z2] #Y blir Z i Staad
        node_index += 1

    return noder

# Setter opp funksjon for å bygge dictionary med noder i mønet med tilhørende koordinater
def dict_noder_mone(n_truss, start, Y, Zm):

    noder = {}
    node_index = start  # Startverdi for dictionary indeksnummerering
    ts = L / (n_truss - 1)

    for i in range(n_truss):
        TSx = ts*i  
        # Node ved mønet: z = Zm = B/2, y = mH
        noder[node_index] = [TSx, Y, Zm]
        node_index += 1

    return noder

# Definerer noder i etasjeskille (ved gesimshøyde) (z=H) 
with open("staad-geometri.txt", "a", encoding="utf-8") as f:
    start = 11
    nodesLOFT = dict_noder(n_truss, start, Y=H, Z1=0, Z2=B)

    # Overstyr første to noder
    nodesLOFT[11] = [0, H, 0]
    nodesLOFT[12] = [0, H, B]

    # Overstyr siste to noder
    keys_sorted = sorted(nodesLOFT.keys())
    second_last = keys_sorted[-2]
    last = keys_sorted[-1]

    nodesLOFT[second_last] = [L, H, 0]
    nodesLOFT[last]        = [L, H, B]

    for i in keys_sorted:
        print(i, *nodesLOFT[i], ";", file=f)
       
# Definerer noder for "takstoler" ved mønet   
if mone == "ja":  
    with open("staad-geometri.txt", "a", encoding="utf-8") as f:
        start = 41
        nodesMONE = dict_noder_mone(n_truss, start, Y=mH, Zm=B/2)
        
        for i, verdier in nodesMONE.items():
            print(i, *verdier, ";", file=f)

#___________________________________________________________________________________________________________________________
# Bygger bjelke-elementer mellom noder og skriver STAAD kommandoer til txt-fil
#___________________________________________________________________________________________________________________________

with open("staad-geometri.txt", "a", encoding="utf-8") as f:
    print("MEMBER INCIDENCES", file=f)

    # Etablerer "start" dictionary for søyler i byggets 4 hjørner, som utvides om flere søyler: starter på 1-serie
    beamsSOYLE = {
        1: [1, 11],
        2: [2, 12],
        3: [3, 13],   # disse to blir overskrevet når n_truss > 2
        4: [4, 14]
    }
    # Finn siste og nest siste nøkkel i nodesLOFT, om n_truss > 2
    if n_truss > 2:
        keys_sorted = sorted(nodesLOFT.keys())
        nest_siste = keys_sorted[-2]
        siste = keys_sorted[-1]

        # Overstyr kun key 3 og key 4 ved n_truss > 2
        beamsSOYLE[3] = [3, nest_siste]
        beamsSOYLE[4] = [4, siste]

    for i, verdier in beamsSOYLE.items():
        print(i, *verdier, ";", file=f)  

    # Bygger bjelker til gesims mellom takstoler / loftsbjelker: starter på 101-serie
    beamsGESIMS = {}
    node_id = sorted(nodesLOFT.keys())

    start_id = 101
    for i in range(2*(n_truss)-2):
        a = node_id[i]
        b = node_id[i + 2]
        beamsGESIMS[start_id] = [a, b]
        start_id += 1
     
    for i, verdier in beamsGESIMS.items():
        print(i, *verdier, ";", file=f)
                
    # Antall loftsbjelker vil avhenge av oppgitt antall takstoler/lofts-bjelker: n_truss, starter på 201-serie 
    node_id = sorted(nodesLOFT.keys())  

    beamsLOFT = {}
    start_id = 201

    for i in range(0, len(node_id), 2):
        a = node_id[i]
        b = node_id[i + 1]
        beamsLOFT[start_id] = [a, b]
        start_id += 1

    for i, verdier in beamsLOFT.items():
        print(i, *verdier, ";", file=f) 

    # Bygger bjelke-elementer langs mønet, starter på 301-serie
    beamsMONE = {}
    if mone == "ja":
        start_id = 301
        node_id = sorted(nodesMONE.keys())

        # Lag glidende par
        for i in range(len(node_id) - 1):
            a = node_id[i]
            b = node_id[i + 1]
            beamsMONE[start_id + i] = [a, b]

        for i in sorted(beamsMONE.keys()):
            print(i, *beamsMONE[i], ";", file=f)

    # Bygger bjelker for taksperrer, basert på antall oppgitt takstoler, starter på 401-serie 
    beamsSPERRE = {}
    if mone == "ja":
        start_id = 401  # fast startverdi for beamsSPERRE keys
        node_id  = sorted(nodesLOFT.keys())  # loftsnoder (taksperrer skl kobles disse til mønenoder)
        
        tm_start = 41
        # Koble hver loftsnode (t) til riktig mønenode (tm)
        # Mønsteret er: 41, 41, 42, 42, 43, 43, osv
        for i, t in enumerate(node_id):
            tm = tm_start + (i // 2)
            beamsSPERRE[start_id + i] = [t, tm]

        for i in sorted(beamsSPERRE.keys()):
           print(i, *beamsSPERRE[i], ";", file=f)

"""
___________________________________________________________________________________________________________________________
Del #3 - Material

Genererer material-data til STAAD.Pro
Samler Staad-kode i filen: staad-material.txt
(Opprinnelig skrevet som selvstendig modul)
___________________________________________________________________________________________________________________________

Denne revisjonen av modulen er "hard-kodet" for S355 og enheter i mm; S355 brukes i så godt som alle tilfeller
I material-variable defineres alle parametre som kreves for MATERIAL i Staad.Pro
"""
# Angir Staad kode for å definere materialene, her kun ett materiale: "ISOTROPIC STEEL_355_NMM2"
material = [
    "DEFINE MATERIAL START",
    "ISOTROPIC STEEL_355_NMM2",
    "E 2.04999e+08",
    "POISSON 0.3",
    "DENSITY 76.9995",
    "ALPHA 6.66667e-06",
    "DAMP 0.03",
    "G 7.88458e+07",
    "TYPE STEEL",
    "STRENGTH FY 354998 FU 469998 RY 1.5 RT 1.2",
    "ISOTROPIC STEEL",
    "E 1.99947e+08",
    "POISSON 0.3",
    "DENSITY 76.8191",
    "ALPHA 6.5e-06",
    "DAMP 0.03",
    "G 7.7221e+07",
    "TYPE STEEL",
    "STRENGTH RY 1.5 RT 1.2",
    "END DEFINE MATERIAL",
]

# Skriver Staad-kode til fil
with open("staad-material.txt", "w", encoding="utf-8") as f:
    for linje in material:
        f.write(linje + "\n")

"""
__________________________________________________________________________________________________________________________
Del #4 - Profiler

I denne versjonen er alle bjelkelementer "hardkodet" til SHS120.
Videre utvikling har som mål å inkludere en GUI for valg av profiltype fra et komplett bibliotek av tilgjengelige profiler

Tilegner valgte profil-typer til bjelke-elementene i dict som lister de ulike gruppene av bjelketyper i geometrimodellen

(Opprinnelig skrevet som selvstendig modul)
_________________________________________________________________________________________________________________________
"""   

# Setter opp funksjon som leser en dict (her gruppe med bjelke-elementer), finner minste og største key: node nummer
# Og skriver Staad kode for elementene i bestemt bjelke gruppe (dict)
# Her harddkodet til SHS120x10 profil
#________________________________________________________________________________________________________________________

def build_profil_list(dict_list):
    if not dict_list:
        return []  # ingenting å gjøre
    i = min(dict_list.keys())
    j = max(dict_list.keys())
    return [f"{i} TO {j} TABLE 'SHS' ST 'SHS 120x10'"]

# Samler alle bjelke-grupper i en samlet dict.
beam_gr = [beamsSOYLE, beamsLOFT, beamsGESIMS]
if mone == "ja":
    beam_gr += [beamsMONE, beamsSPERRE]

# Bruker funksjonen på bjelke-gruppene og skriver Staad-kode for hver gruppe
profiler = []
for elem in beam_gr:
    profiler.extend(build_profil_list(elem))

with open("staad-profiler.txt", "w", encoding="utf-8") as f:
    print("MEMBER PROPERTY 'EUROPE (EN 2023).DB3'", file=f)
    for linje in profiler:
        f.write(linje + "\n")
#_____________________________________________________________________________________________________________________________
# Tilegner (eventuelle CONSTANTS, ingen her) og tidligere definert material-type S355 til alle profiler
# Disse må defineres på dette stedet i en Staad-kommando-fil, rekkefølge-krav
#_____________________________________________________________________________________________________________________________

with open("staad-profiler.txt", "a", encoding="utf-8") as f:
    print("CONSTANTS", file=f)
    print("MATERIAL STEEL_355_NMM2 ALL", file=f)

"""
_____________________________________________________________________________________________________________________________
Del #5 - Support

Genererer kommandoer til STAAD.Pro for opplagerbetingelser (SUPPORTS)
Samler Staad-kode i filen: staad-support.txt

I denne revisjonen er alle opplagerbetingelser "hard-kodet" til fritt-opplagret (PINNED) ved søylene
Fritt-opplager vil være riktig for alle enkle stålbygg

(Opprinnelig skrevet som selvstendig modul)
_______________________________________________________________________________________________________________________________

"""
# Definerer en funksjon som oppretter en dictionary av alle noder som skal ha opplagerbetingelser
def build_support_list(nodes):
 
    # Lager en SUPPORTS-liste der hver nodeindex får 'i PINNED'.
    support = ["SUPPORTS"]

    # Legger verdien PINNED til listen support, for hver keys i dictionary "nodes", som ved bruk vil være alle fundament-noder
    for i in nodes.keys():
        support.append(f"{i} PINNED")
    return support

# Anvender funksjonen som definerer PINNED-opplager på fundament-nodene
support = build_support_list(nodes0)

# Skriver Staad-kode til fil
with open("staad-support.txt", "w", encoding="utf-8") as f:
    for linje in support:
        f.write(linje + "\n")
"""
____________________________________________________________________________________________________________________________
Del #6 - Laster

Definerer laster og genererer last-input-kommandoer til STAAD.Pro 
Samler Staad-kode i filen: staad-load.txt
(Opprinnelig skrevet som selvstendig modul)
___________________________________________________________________________________________________________________________

GUI for bruker på verdi i kg/m2 på snølaster
"""
snolast = None
# Lager funksjon som kan kalles av tkinter
def lagre_snolast():
    global snolast
    snolast_txt = float(entry.get())  # Henter brukerens  verdi på snølast
   
    # Konverterer tekst til flyttall 
    snolast = parse_float(snolast_txt)
    if snolast is None or snolast <= 0:
        messagebox.showerror("Feil", "Oppgi gyldig, positiv verdi på snølast i kg/m^3.")
        return
    root.destroy()                # Lukker GUI-vinduet

# Genererer GUI-vindu
root = tk.Tk()
root.title("Snølast-input")
root.geometry("300x150")  # Størrelse på GUI
# Angir tekst + inputfelt til GUIet
label = tk.Label(root, text="Angi snølast (kg/m²):")
label.pack()
# Oppretter inputfeltet
entry = tk.Entry(root)
entry.pack()

# Knapp som lagrer variabelen
button = tk.Button(root, text="Lagre", command=lagre_snolast)
button.pack(pady=10)

# Sender inn data når bruker trykker "Enter", isteden for å trykke inn "Send inn"
root.bind("<Return>", lambda _e: lagre_snolast())

# Holder GUIet åpent
root.mainloop()

#______________________________________________________________________________________________________________________
# Definerer last-tilfelle "LOAD 1" med design-last (her kun snø + egenvekt) og tilegner disse til noder
#______________________________________________________________________________________________________________________

# Skriver til filen "staad-load.txt" for å definere lasttilfelle LOAD 1 - designlaster, her snø + egenvekt (+10%)
with open("staad-load.txt", "w", encoding="utf-8") as f:
    print("LOAD 1 LOADTYPE None  TITLE Design", file=f)
    print("MEMBER LOAD", file=f)

# Definerer en funksjon som oppretter en dictionary av alle elementer som skal ha linjelast
def build_load_list(nodes):
    loads = []
    
    # Henter min og maks key for å definere last fra node til node (ende-noder for tak-elementer)
    i = min(nodes.keys())
    j = max(nodes.keys())

    # Skriver Staad-kode for vertikal last, gitt av brukerinput "snolast", fra node i til node j
    last = -snolast * 9.81 / 1000 # Konverterer fra kg til kN
    loads.append(f"{i} TO {j} UNI GY {last}")
    return loads

    

if mone == "ja":
    # Legger last på taksperrer 
    loads = build_load_list(beamsSPERRE)
    # Skriver Staad-kode til fil
    with open("staad-load.txt", "a", encoding="utf-8") as f:
        for linje in loads:
            f.write(linje + "\n")
        print("SELFWEIGHT Y -1.1", file=f)
    
else:
    # Legger last på indre tak-bjelker
    loads = build_load_list(beamsLOFT)
    # Skriver Staad-kode til fil
    with open("staad-load.txt", "a", encoding="utf-8") as f:
        for linje in loads:
            f.write(linje + "\n")
        print("SELFWEIGHT Y -1.1", file=f)           
    

#___________________________________________________________________________________________________________________________
# Definerer last-tilfelle "LOAD 2" med kun egenvekt (+10%) som tilegnes alle elementer i modellen
#____________________________________________________________________________________________________________________________
  
# Skriver til filen "staad-load.txt" for å definere lasttilfelle LOAD 2 - egenvekt med 10% ekstra (contingency)
with open("staad-load.txt", "a", encoding="utf-8") as f:
    print("LOAD 2 LOADTYPE None  TITLE Egenvekt", file=f)
    print("SELFWEIGHT Y -1.1", file=f)

"""
_____________________________________________________________________________________________________________________________
Del #7 - Kodesjekk

Genererer kommandoer for kodesjekk etter EN1993-1-1 i STAAD.Pro og definerer material-faktorer GM0 til GM2
Samler Staad-kode i filen: staad-kode.txt
(Opprinnelig skrevet som selvstendig modul)
_____________________________________________________________________________________________________________________________
"""
# Skriver Staad-kode som definerer kodesjekk, variablen "kode" inneholde nødvendige kommandoer
# Dette avslutter kommandoene til Staad filen med "FINISH"
kode = [
    "PERFORM ANALYSIS PRINT STATICS CHECK",
    "PARAMETER 1",
    "CODE EN 1993-1-1:2005",
    "TRACK 2 ALL",
    "GM0 1.05 ALL",
    "GM1 1.05 ALL",
    "GM2 1.25 ALL",
    "CHECK CODE ALL",
    "FINISH"
]

# Skriver Staad-kode for kodesjekk til fil
with open("staad-kode.txt", "w", encoding="utf-8") as f:
    for linje in kode:
        f.write(linje + "\n")

"""
_________________________________________________________________________________________________________________________________________
Del #8 - STD-fil

Skriver tilslutt alle kommandoer fra txt-filer til en komplett input STD-fil

Filen kan også åpnes i feks Notepad (en tekstleser)
(Opprinnelig del av hoved-script som leser de øvrige moduler)
_________________________________________________________________________________________________________________________________________
"""
input_filer = ["staad-geometri.txt", "staad-material.txt",  "staad-profiler.txt", "staad-support.txt", "staad-load.txt", "staad-kode.txt"]
kommando_fil = "staad-command.std"

# Åpner filen staad-command.std for å skrives til
with open(kommando_fil, "w", encoding="utf-8") as staad_input:
    
    # Leser innholdet i hver an filene in "input-filer" og skriver dette til output-file: staad-command.std
    for txt_filer in input_filer:
        with open(txt_filer, "r", encoding="utf-8") as innfiler:
            staad_input.write(innfiler.read())

"""
_________________________________________________________________________________________________________________________________________
Del #9 - Plotter noder - dvs knutepunkter mellom bjelke-elementer

Siden oppgaven krever plotting så plottes alle noder definert for bygget
Et bjelkelement har en node i hver ende. 

Plottet vil illustrere byggets form og kan brukes som en kontroll for at bruker-input er som forventet.
_________________________________________________________________________________________________________________________________________
"""
# Importere matplotlib for plotting av knutepunkter (noder i Staad-kode)
import matplotlib.pyplot as plt

# Setter opp en funksjon som plotter basert på en dict med noder (keys: noder-nummer og verdier: x,y,z-koordinater)
def plot_noder(d):
    fig = plt.figure(figsize=(12, 5)) # Setter størrelse på plot
    # Plotter i både ISO view og sett ned x-aksen
    ax_iso = fig.add_subplot(1, 2, 1, projection='3d') 
    ax_x   = fig.add_subplot(1, 2, 2, projection='3d')
   
    for key, koord in d.items():
        x, y, z = float(koord[0]), float(koord[2]), float(koord[1])
        color = "blue"
        # ISO-visningen
        ax_iso.scatter([x], [y], [z], s=30, color=color)
        # X-akse-visningen
        ax_x.scatter([x], [y], [z], s=30, color=color)
        
    # ISO-view
    ax_iso.set_xlabel("Lengde [m]")
    ax_iso.set_ylabel("Bredde [m]")
    ax_iso.set_zlabel("Høyde [m]")
    ax_iso.set_title("Byggets knutepunkter (ISO)")
    ax_iso.grid(True)

    # X-akse-visning 
    ax_x.set_xlabel("Lengde [m]")
    ax_x.set_ylabel("Bredde [m]")
    ax_x.set_zlabel("Høyde [m]")
    ax_x.set_title("Byggets knutepunkter – sett langs X-aksen")
    ax_x.view_init(elev=0, azim=0)  # sett langs +X (Y–Z-projeksjon)
    ax_x.grid(True)
    plt.tight_layout()
    plt.show()

# Samler alle noder i en felles dict
alle_noder = {**nodes0, **nodesLOFT, **(nodesMONE if mone == "ja" else {})}
# Bruker funksjonen for å plotte nodene / knutepunktene
plot_noder(alle_noder)